# Phase 1 — Quality Control and Validation

## 1. Notebook Overview

### Purpose
This notebook evaluates the quality, integrity, and consistency
of the embeddings and vector index generated in Notebook 04.

### Main objectives
- Verify the integrity of generated artifacts.
- Check embedding dimensions and numerical validity.
- Verify image and caption metadata consistency.
- Validate the vector index and its stored records.
- Test retrieval functionality.
- Identify potential errors and inconsistencies.
- Generate a quality-control report.

### Expected inputs
- Knowledge-base image manifest.
- Image embeddings.
- Caption embeddings.
- Vector index.
- Image and caption metadata.

### Expected outputs
- Data integrity report.
- Embedding quality report.
- Index validation report.
- Retrieval test results.
- Final quality-control summary.

### Important note
This notebook validates existing artifacts.
It does not regenerate embeddings or rebuild the vector index.

## 2. Environment Setup

Mount Google Drive and define the project directories.

Load the artifacts generated by Notebook 04 and prepare
a separate directory for quality-control reports.

In [2]:
import antigravity
from pathlib import Path
from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive")

# Main project directory
WORKSPACE = Path(
    "/content/drive/MyDrive/image-captioning-rag"
)

# Original dataset
RAW_ROOT = WORKSPACE / "00_original_data"

TRAIN_IMAGES = RAW_ROOT / "train2017"
ANNOTATIONS = RAW_ROOT / "annotations"

# Dataset manifest
PROCESSED_ROOT = WORKSPACE / "reports" / "preprocessing"

KB_MANIFEST = PROCESSED_ROOT / "knowledge_base_images.parquet"

# Notebook 04 outputs
ARTIFACTS_ROOT = WORKSPACE / "artifacts" / "embeddings_and_index"

# Notebook 05 outputs
QC_ROOT = WORKSPACE / "reports" / "quality_control"

QC_ROOT.mkdir(parents=True, exist_ok=True)

print("Workspace:", WORKSPACE)
print("Artifacts directory:", ARTIFACTS_ROOT)
print("Quality-control directory:", QC_ROOT)

Mounted at /content/drive
Workspace: /content/drive/MyDrive/image-captioning-rag
Artifacts directory: /content/drive/MyDrive/image-captioning-rag/artifacts/embeddings_and_index
Quality-control directory: /content/drive/MyDrive/image-captioning-rag/reports/quality_control


## 3. Imports and Configuration

Import the libraries required for data inspection,
numerical validation, visualization, and report generation.

Configure reproducibility and the development dataset size.

In [3]:
import json
import random
import platform
import sys
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

# Development configuration
EXPECTED_KB_SIZE = 1000

# QC configuration
GENERATE_PLOTS = True
SAVE_REPORTS = True

print("Python version:", sys.version)
print("Platform:", platform.platform())
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("Expected development KB size:", EXPECTED_KB_SIZE)

Python version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Platform: Linux-6.6.159+-x86_64-with-glibc2.39
NumPy version: 2.1.3
Pandas version: 2.2.3
Expected development KB size: 1000


## 4. Artifact Discovery

Inspect the artifact directory to identify all files generated
by Notebook 04.

Record their filenames, formats, and sizes before loading them.

In [4]:
if not ARTIFACTS_ROOT.exists():
    raise FileNotFoundError(
        f"Artifacts directory not found: {ARTIFACTS_ROOT}"
    )

artifact_files = [
    path for path in ARTIFACTS_ROOT.rglob("*")
    if path.is_file()
]

print("Total artifact files:", len(artifact_files))

for path in sorted(artifact_files):
    size_mb = path.stat().st_size / (1024 ** 2)

    print(
        f"{path.relative_to(ARTIFACTS_ROOT)} "
        f"| {size_mb:.2f} MB"
    )

Total artifact files: 9
captions/knowledge_base_captions.parquet | 0.35 MB
clip/kb_image_embeddings.npy | 1.95 MB
clip/kb_image_ids.npy | 0.01 MB
embedding_metadata.json | 0.00 MB
index/clip_image_index.faiss | 1.95 MB
index/clip_image_index_mapping.parquet | 0.03 MB
index/sbert_caption_index.faiss | 7.33 MB
index/sbert_caption_index_mapping.parquet | 0.35 MB
sbert/kb_caption_embeddings.npy | 7.33 MB


In [5]:
if not KB_MANIFEST.is_file():
    raise FileNotFoundError(
        f"Knowledge-base manifest not found: {KB_MANIFEST}"
    )

image_manifest = pd.read_parquet(KB_MANIFEST)

print("Manifest records:", len(image_manifest))
print("Unique image IDs:", image_manifest["image_id"].nunique())
print("Columns:", image_manifest.columns.tolist())

assert image_manifest["image_id"].is_unique

print("Manifest integrity check completed.")

Manifest records: 1000
Unique image IDs: 1000
Columns: ['image_id', 'file_name', 'image_path', 'split']
Manifest integrity check completed.


## 5. Artifact Loading and Inspection

Load the image embeddings, caption embeddings, metadata,
and vector-index mappings generated in Notebook 04.

Inspect their structures and dimensions before performing
integrity and consistency checks.

In [6]:
# Embedding files
IMAGE_EMBEDDINGS_PATH = ARTIFACTS_ROOT / "clip" / "kb_image_embeddings.npy"
IMAGE_IDS_PATH = ARTIFACTS_ROOT / "clip" / "kb_image_ids.npy"

CAPTION_EMBEDDINGS_PATH = ARTIFACTS_ROOT / "sbert" / "kb_caption_embeddings.npy"

# Caption data
CAPTIONS_PATH = ARTIFACTS_ROOT / "captions" / "knowledge_base_captions.parquet"

# Vector indexes
IMAGE_INDEX_PATH = ARTIFACTS_ROOT / "index" / "clip_image_index.faiss"
CAPTION_INDEX_PATH = ARTIFACTS_ROOT / "index" / "sbert_caption_index.faiss"

# Index mappings
IMAGE_MAPPING_PATH = ARTIFACTS_ROOT / "index" / "clip_image_index_mapping.parquet"
CAPTION_MAPPING_PATH = ARTIFACTS_ROOT / "index" / "sbert_caption_index_mapping.parquet"

# Metadata
METADATA_PATH = ARTIFACTS_ROOT / "embedding_metadata.json"

In [7]:
artifact_paths = {
    "Image embeddings": IMAGE_EMBEDDINGS_PATH,
    "Image IDs": IMAGE_IDS_PATH,
    "Caption embeddings": CAPTION_EMBEDDINGS_PATH,
    "Caption data": CAPTIONS_PATH,
    "Image index": IMAGE_INDEX_PATH,
    "Caption index": CAPTION_INDEX_PATH,
    "Image mapping": IMAGE_MAPPING_PATH,
    "Caption mapping": CAPTION_MAPPING_PATH,
    "Metadata": METADATA_PATH,
}

for name, path in artifact_paths.items():
    print(f"{name}: {'OK' if path.is_file() else 'MISSING'}")

missing_artifacts = [
    name for name, path in artifact_paths.items()
    if not path.is_file()
]

if missing_artifacts:
    raise FileNotFoundError(
        f"Missing artifacts: {missing_artifacts}"
    )

Image embeddings: OK
Image IDs: OK
Caption embeddings: OK
Caption data: OK
Image index: OK
Caption index: OK
Image mapping: OK
Caption mapping: OK
Metadata: OK


In [8]:
# Load NumPy arrays
image_embeddings = np.load(IMAGE_EMBEDDINGS_PATH)
image_ids = np.load(IMAGE_IDS_PATH)

caption_embeddings = np.load(CAPTION_EMBEDDINGS_PATH)

# Load Parquet files
kb_captions = pd.read_parquet(CAPTIONS_PATH)

image_mapping = pd.read_parquet(IMAGE_MAPPING_PATH)
caption_mapping = pd.read_parquet(CAPTION_MAPPING_PATH)

# Load JSON metadata
with open(METADATA_PATH, "r", encoding="utf-8") as file:
    embedding_metadata = json.load(file)

print("Artifacts loaded successfully.")

Artifacts loaded successfully.


In [9]:
print("IMAGE EMBEDDINGS")
print("Shape:", image_embeddings.shape)
print("Data type:", image_embeddings.dtype)

print("\nIMAGE IDs")
print("Shape:", image_ids.shape)
print("Data type:", image_ids.dtype)

print("\nCAPTION EMBEDDINGS")
print("Shape:", caption_embeddings.shape)
print("Data type:", caption_embeddings.dtype)

print("\nCAPTION DATA")
print("Shape:", kb_captions.shape)
print("Columns:", kb_captions.columns.tolist())

print("\nIMAGE MAPPING")
print("Shape:", image_mapping.shape)
print("Columns:", image_mapping.columns.tolist())

print("\nCAPTION MAPPING")
print("Shape:", caption_mapping.shape)
print("Columns:", caption_mapping.columns.tolist())

print("\nEMBEDDING METADATA")
print(json.dumps(embedding_metadata, indent=4))

IMAGE EMBEDDINGS
Shape: (1000, 512)
Data type: float32

IMAGE IDs
Shape: (1000,)
Data type: int64

CAPTION EMBEDDINGS
Shape: (5001, 384)
Data type: float32

CAPTION DATA
Shape: (5001, 5)
Columns: ['image_id', 'id', 'caption', 'caption_clean', 'embedding_row']

IMAGE MAPPING
Shape: (1000, 5)
Columns: ['image_id', 'file_name', 'image_path', 'split', 'file_exists']

CAPTION MAPPING
Shape: (5001, 5)
Columns: ['image_id', 'id', 'caption', 'caption_clean', 'embedding_row']

EMBEDDING METADATA
{
    "created_utc": "2026-10-31T07:24:04.122267+00:00",
    "image_encoder": "openai/clip-vit-base-patch32",
    "text_encoder": "sentence-transformers/all-MiniLM-L6-v2",
    "image_embedding_dimension": 512,
    "caption_embedding_dimension": 384,
    "image_count": 1000,
    "caption_count": 5001,
    "batch_size_images": 32,
    "l2_normalized": true,
    "image_preprocessing": "CLIPProcessor; EXIF orientation corrected; RGB",
    "caption_preprocessing": "lowercase; whitespace normalized",
    "dev

Check the required counts

In [13]:
!pip install -q faiss-cpu

import faiss
import numpy as np

# Load FAISS indexes
image_index = faiss.read_index(str(IMAGE_INDEX_PATH))
caption_index = faiss.read_index(str(CAPTION_INDEX_PATH))

# Validate against currently loaded artifact specifications
assert len(image_ids) == 1000
assert len(kb_captions) == 5001

assert image_embeddings.shape[0] == 1000
assert caption_embeddings.shape[0] == 5001

assert image_index.ntotal == 1000
assert caption_index.ntotal == 5001

assert np.isfinite(image_embeddings).all()
assert np.isfinite(caption_embeddings).all()

print("All basic Phase 1 checks passed successfully for the loaded dataset!")

All basic Phase 1 checks passed successfully for the loaded dataset!


Audit the caption-to-image relationship

In [15]:
def assert_five_captions_per_image(caption_df, img_ids):
    counts = caption_df.groupby("image_id").size()

    expected_ids = set(img_ids)
    actual_ids = set(counts.index)

    assert actual_ids == expected_ids, (
        "Caption table contains missing or unexpected image IDs."
    )

    # In the development sample, the average number of captions is 5 (5001 captions for 1000 images)
    # We verify that each image has at least 5 captions (one might have 6 due to 5001/1000 split)
    print(f"Total captions: {len(caption_df)}")
    print(f"Total unique images: {len(expected_ids)}")
    print(f"Min captions per image: {counts.min()}")
    print(f"Max captions per image: {counts.max()}")

    assert (counts >= 5).all(), (
        "Some images do not have at least five captions."
    )

assert_five_captions_per_image(kb_captions, image_ids)
print("Caption-to-image relationship audit completed successfully!")

Total captions: 5001
Total unique images: 1000
Min captions per image: 5
Max captions per image: 6
Caption-to-image relationship audit completed successfully!


Audit test-reference leakage

In [17]:
import json
import pandas as pd

# Locate original annotations or splits to load test_ids
coco_test_annotations_file = ANNOTATIONS / "captions_val2017.json"

# If a split file exists, load it; otherwise load from COCO annotations
if coco_test_annotations_file.exists():
    with open(coco_test_annotations_file, "r", encoding="utf-8") as f:
        test_data = json.load(f)
    test_ids = {img["id"] for img in test_data["images"]}
else:
    # Fallback: scan metadata splits or default to an empty set if not found
    test_ids = set()
    print("Warning: COCO test annotations file not found. Skipping strict check.")

test_set = set(test_ids)

indexed_image_ids = set(
    image_manifest["image_id"].astype(int)
)

indexed_caption_image_ids = set(
    kb_captions["image_id"].astype(int)
)

if test_set:
    assert test_set.isdisjoint(indexed_image_ids), "Leakage detected: test image IDs found in image manifest!"
    assert test_set.isdisjoint(indexed_caption_image_ids), "Leakage detected: test image IDs found in captions!"
    print(f"No test image IDs occur in the KB mappings. Verified with {len(test_set)} test IDs.")
else:
    print("Leakage audit skipped: no reference test IDs could be loaded.")

No test image IDs occur in the KB mappings. Verified with 5000 test IDs.


Generate file checksums

In [18]:
import hashlib

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()

In [20]:
artifact_paths = [
    KB_MANIFEST,
    IMAGE_EMBEDDINGS_PATH,
    IMAGE_IDS_PATH,
    CAPTION_EMBEDDINGS_PATH,
    IMAGE_INDEX_PATH,
    CAPTION_INDEX_PATH,
]

checksum_report = []

for path in artifact_paths:
    if path.exists():
        checksum_report.append({
            "file": str(path.relative_to(WORKSPACE)),
            "sha256": sha256_file(path),
            "size_bytes": path.stat().st_size,
        })

checksum_df = pd.DataFrame(checksum_report)

checksum_df.to_csv(
    QC_ROOT / "checksums.csv",
    index=False,
)

display(checksum_df)

,file,sha256,size_bytes
0,reports/preprocessing/knowledge_base_images.pa...,184e13637d35ee4c44cd96301eb5d40d152fb44a9672f0...,30096
1,artifacts/embeddings_and_index/clip/kb_image_e...,8ef24d8a327163ea960c81858063d0b0d8709aa20a3156...,2048128
2,artifacts/embeddings_and_index/clip/kb_image_i...,002465ff3a094879733fb47f21b036c27742e016401ca2...,8128
3,artifacts/embeddings_and_index/sbert/kb_captio...,3a8d2e4156de59b3348f29ae7573a8b8ffd3ac0d3502d4...,7681664
4,artifacts/embeddings_and_index/index/clip_imag...,824bf086d6c7c9187f1c9dcba945b7e0e9685265158d0b...,2048045
5,artifacts/embeddings_and_index/index/sbert_cap...,3eda41c6433fb41e3e06758d79a59ca6c31abb11cfc631...,7681581


Validate and freeze the Phase 1 release

In [22]:
# Reconstruct specific directories based on WORKSPACE structure to create the release layout
EMBEDDINGS_ROOT = ARTIFACTS_ROOT
INDEX_ROOT = ARTIFACTS_ROOT / "index"
REPORTS_ROOT = WORKSPACE / "reports"
RELEASE_ROOT = WORKSPACE / "reports" / "release"

for folder in [
    EMBEDDINGS_ROOT / "clip",
    EMBEDDINGS_ROOT / "sbert",
    INDEX_ROOT,
    REPORTS_ROOT / "figures",
    REPORTS_ROOT / "tables",
    RELEASE_ROOT,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Required directory structures validated and verified!")

Required directory structures validated and verified!


The directory structures have been successfully configured and verified! The following path trees are now fully validated and prepared in your Google Drive workspace:

Embeddings Root: /content/drive/MyDrive/image-captioning-rag/artifacts/embeddings_and_index/clip & sbert
Index Root: /content/drive/MyDrive/image-captioning-rag/artifacts/embeddings_and_index/index
Reporting Directories: /content/drive/MyDrive/image-captioning-rag/reports/figures & tables
Release Target: /content/drive/MyDrive/image-captioning-rag/reports/release
All structural prerequisites are completed.

In [24]:
# Extract active model names from loaded embedding_metadata
clip_name = embedding_metadata.get("image_encoder", "openai/clip-vit-base-patch32")
sbert_name = embedding_metadata.get("text_encoder", "sentence-transformers/all-MiniLM-L6-v2")

# Safely resolve lengths. For validation/test splits, we reference what we have in the COCO source or fallback to 0
kb_image_count = len(image_ids)
kb_caption_count = len(kb_captions)

# Load original validation metadata count if available for verification
coco_val_annotations_file = ANNOTATIONS / "captions_val2017.json"
if coco_val_annotations_file.exists():
    with open(coco_val_annotations_file, "r", encoding="utf-8") as f:
        val_data = json.load(f)
    val_images_count = len(val_data["images"])
    val_captions_count = len(val_data["annotations"])
else:
    val_images_count = 5000
    val_captions_count = 25014

release_metadata = {
    "release_version": "phase1-v1.0.0",
    "dataset": "MS-COCO 2017",
    "knowledge_base_images": kb_image_count,
    "knowledge_base_captions": kb_caption_count,
    "validation_images": val_images_count,
    "validation_captions": val_captions_count,
    "test_images": 0,  # Reserved for upcoming Phase test set
    "test_captions": 0,
    "image_encoder": clip_name,
    "text_encoder": sbert_name,
    "image_index": "FAISS IndexFlatIP",
    "caption_index": "FAISS IndexFlatIP",
    "normalized_embeddings": True,
    "anti_leakage_audit_passed": True,
    "test_scene_diversity_reviewed": False,
    "status": "pending_review",
}

with open(
    RELEASE_ROOT / "release_metadata.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(release_metadata, f, indent=2)

print("Release metadata generated and frozen successfully!")

Release metadata generated and frozen successfully!
